# Ejemplo de DeepFakes usando Wan2.2-Animate en Colab con GPU T4

Animar un personaje dado por una **imagen** siguiendo el movimiento de un **video** de referencia, guiado por un **prompt**.

## Detalles de ajustes para Colab

El modelo tiene 17.3-billion parámetros y es un transformer de difusión. En su forma nativa requiere **34.5 GB para sus pesos en formato bf16** y está pensado para GPUs de 80 GB. La cuenta gratuita de Colab T4 tiene una GPU de **15 GB de VRAM y cerca de 12.7 GB de RAM**. Para permitir su ejecución en Google Colab se usó estrategias que reducen ligeramente la capacidad del modelo, que cambian piezas por otras más "livianas" y que cambian el cómputo paralelo por ejecución secuencial.

## 1. Confirmar acceso a GPU

Si la ejecución indica que no tienes GPU, debes ir a:

**Entorno de ejecución &rarr; Cambiar tipo de entorno de ejecución &rarr; T4 GPU**, y volver a ejecutar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


## 2. Descargamos y descomprimimos los archivos necesarios

In [ ]:
!mkdir -p assets
!if [ ! -f wan_t4_colab.zip ]; then wget -q -O wan_t4_colab.zip "https://www.dropbox.com/scl/fi/70ys3ccgvcoqiftpmec5h/wan_t4_colab.zip?rlkey=dewo4olt172xeibkqgbqyl8r7&dl=1"; fi
!if [ ! -f assets/sample_face.png ]; then wget -q -O assets/sample_face.png "https://www.dropbox.com/scl/fi/f4v9phd8nz66ocw9z8zb1/sample_face.png?rlkey=0mu6h44z3ubyfx3evlpgb0blu&dl=1"; fi
!if [ ! -f assets/sample_video.mp4 ]; then wget -q -O assets/sample_video.mp4 "https://www.dropbox.com/scl/fi/8m7z9yme8vv8z5bhrn3dr/sample_video.mp4?rlkey=r8f1c7gwq5ombhz0l56bbhupn&dl=1"; fi

In [ ]:
import os, sys, zipfile, pathlib, shutil

ZIP_FILE = "/content/wan_t4_colab.zip"
PROJECT  = "/content/wan_2_2"

archive = pathlib.Path(ZIP_FILE)
if not archive.exists():
    raise SystemExit(
        str(archive) + ' is not there. Open the Files pane (the folder icon on '
        'the left), drop wan_t4_colab.zip into /content, and re-run this cell.')

project = pathlib.Path(PROJECT)
for owned in ('wan_t4', 'scripts', 'docs', 'configs'):
    shutil.rmtree(project / owned, ignore_errors=True)
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(PROJECT)

root = project
if not (root / 'wan_t4' / 'run.py').exists():
    found = next((p for p in root.rglob('wan_t4/run.py')), None)
    if found is None:
        listing = sorted(p.name for p in root.iterdir())
        raise SystemExit(
            'Could not find wan_t4/run.py under ' + PROJECT + '. The archive should '
            'contain wan_t4/, scripts/ and requirements.txt at its top level. '
            'Found instead: ' + str(listing))
    root = found.parent.parent
    print('note: sources were nested one level deeper; using', root)

os.chdir(root)
sys.path.insert(0, str(root))

archive: /content/wan_t4_colab.zip (100 KB)
working directory: /content/wan_2_2
contents: ['docs', 'requirements.txt', 'scripts', 'wan_t4']

OK - code is in place.


## 3. Instalar dependencias faltantes

In [ ]:
!python scripts/colab_setup.py

Environment check
----------------------------------------------------------------------

GPU:
  [ok  ] torch: 2.11.0+cu128
  [ok  ] gpu: Tesla T4, sm_75, 14.6 GB
  [ok  ] arch: Turing sm_75 -- the intended target

Memory:
  [ok  ] system RAM: 11.2 GB free of 12.7 GB
  [ok  ] disk: 65.7 GB free (need ~20 GB for weights)

Packages:
  [ok  ] diffusers: 0.39.0
  [ok  ] transformers: 5.13.1
  [warn] gguf: missing
  [warn] onnxruntime: missing
  [ok  ] opencv: 5.0.0
  [ok  ] imageio-ffmpeg: 0.6.0
  [ok  ] peft: 0.19.1
  [ok  ] WanAnimateTransformer3DModel: available
  [ok  ] GGUFQuantizationConfig: available

Installing: gguf onnxruntime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 76.5 MB/s eta 0:00:00

Re-checking after install:
  [ok  ] diffusers: 0.39.0
  [ok  ] transformers: 5.13.1
  [ok  ] gguf: present
  [ok  ] onnxruntime: 1.28.0
  [ok  ] opencv: 5.0.0
  [ok  ] imageio-ffmpeg: 0.6.0
  [ok  ] 

## 4. Seleccionar el modo ("preset")

El modelo se puede ejecutar en varios modos. Mientras más avanzado mejor calidad, pero estresa más a la GPU

In [ ]:
!python -m wan_t4.run --list-presets

Presets:
  t4-draft       480x480 33f  4 steps  q3_k_m  distill x1.0     ~4-5 min/segment
                 Smallest useful configuration. Use this first to prove the whole pipeline runs end to end before spending real time.
  t4-balanced    640x384 49f  6 steps  q3_k_m  distill x1.0     ~5-7 min/segment
                 The recommended default. Widescreen, ~3 seconds per segment at 16 fps, and comfortable headroom on a 16 GB card.
  t4-quality     832x480 49f  6 steps  q3_k_m  distill x1.0     ~9-12 min/segment
                 Native 480p at the model's trained aspect -- the best quality that reliably fits. Q4_K_M is deliberately NOT used here: at this resolution and frame count it needs 14.15 GB against a 14.0 GB budget, so it is predicted to OOM. Add --dit-quant q4_k_m --frame-num 45 if you want to try it. refert_num=5 reduces drift across segments at ~7% extra cost per output frame.
  t4-max         832x480 77f  6 steps  q3_k_m  distill x1.0     ~17-19 min/segment
                 

## 5. Descarga de pesos

Alrededor de **20 GB** totales.


In [ ]:
!python scripts/fetch_preprocess_models.py

from huggingface_hub import snapshot_download, hf_hub_download

DIT_QUANT = 'Q3_K_M'   # Q4_K_M es mejor, pero pesa 10.7 GB en lugar de 8.0 GB
hf_hub_download('QuantStack/Wan2.2-Animate-14B-GGUF',
                f'Wan2.2-Animate-14B-{DIT_QUANT}.gguf',
                local_dir='weights/dit')
hf_hub_download('Comfy-Org/Wan-Animate-2',
                'loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors',
                local_dir='weights')
snapshot_download('Wan-AI/Wan2.2-Animate-14B-Diffusers',
                  allow_patterns=['text_encoder/*'])
print('weights ready')

dest   : /content/wan_2_2/weights/preprocess
models : dwpose + yolox_l
size   : 334.9 MB
disk   : 65.6 GB free

  dw-ll_ucoco_384.onnx     downloading 128.2 MB from yzd-v/DWPose

dw-ll_ucoco_384.onnx: downloading bytes:  59% 79.4M/134M [00:01<00:00, 113MB/s, 5.47MB/s  ] 
dw-ll_ucoco_384.onnx: downloading bytes:  89% 120M/134M [00:01<00:00, 104MB/s, 10.6MB/s  ]
dw-ll_ucoco_384.onnx: downloading bytes: 100% 127M/127M [00:01<00:00, 65.6MB/s, 11.9MB/s  ]
dw-ll_ucoco_384.onnx: reconstructing file: 100% 134M/134M [00:01<00:00, 69.2MB/s, 12.6MB/s  ]
  yolox_l.onnx             downloading 206.7 MB from hr16/yolox-onnx

yolox_l.onnx: downloading bytes:  45% 98.3M/217M [00:01<00:01, 111MB/s, 8.21MB/s  ]
yolox_l.onnx: downloading bytes:  53% 115M/217M [00:01<00:00, 121MB/s, 9.27MB/s  ] 
yolox_l.onnx: downloading bytes:  63% 136M/217M [00:01<00:00, 143MB/s, 10.7MB/s  ]
yolox_l.onnx: downloading bytes:  82% 178M/217M [00:02<00:00, 142MB/s, 14.8MB/s  ]
yolox_l.onnx: downloading bytes: 100% 204M/204M

Wan2.2-Animate-14B-Q3_K_M.gguf: reconstructing file:   0%|          |  0.00B / 8.63GB            

Wan2.2-Animate-14B-Q3_K_M.gguf: downloading bytes:           |  0.00B            

loras/lightx2v_I2V_14B_480p_cfg_step_dis(…): reconstructing file:   0%|          |  0.00B /  738MB            

loras/lightx2v_I2V_14B_480p_cfg_step_dis(…): downloading bytes:           |  0.00B            

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

weights ready


## 6. Seleccionar los inputs e hiperparámetros

- **`/content/assets/sample_video.mp4`** &mdash; el video del movimiento. Una sola persona, claramente visible. Que todo el cuerpo sea visible si se quiere usar movimiento de cuerpo completo. Se recomienda cideos cortos pues el costo escala con la longitud del video.
- **`/content/assets/sample_face.png`** &mdash; La imagen de referencia, que identifica al personaje a ser animado. Cara frontal con extremidades visibles funcionan mejor. La pose se ancla al primer frame.
- El **prompt** es optional para guiar al modelo. Para desactivarlo se recomienda usar una oración genérica.

In [ ]:
import pathlib

VIDEO = "/content/assets/sample_video.mp4"   # movimiento
IMAGE = "/content/assets/sample_face.png"    # personaje
PRESET = 't4-draft'
PROMPT = 'a person dancing'
MAX_FRAMES = 33     # recortar el video. Se recomienda ampliar la duración poco a poco

video: /content/assets/sample_video.mp4  (14.9 MB)
image: /content/assets/sample_face.png  (0.2 MB)


## 7. Ejecutar el plan en marcha blanca, para verificar que cabe en la VRAM

`--dry-run` imprime la predicción de uso de memoria, y un tiempo estimado.

In [ ]:
!python -m wan_t4.run \
    --video "{VIDEO}" --image "{IMAGE}" \
    --prompt "{PROMPT}" --preset {PRESET} \
    --max-frames {MAX_FRAMES} \
    --out outputs/result.mp4 --dry-run

[run] preset 't4-draft' -> 480x480, 33f/segment, 4 steps, q3_k_m
[run] driving video: 3840x2160, 142 frames @ 24.0 fps
[run] 480x480 x 33f -> latents [10, 60, 60] = 9,000 tokens
[run] weights 8.04 GB + activations 0.58 GB + attention 0.34 GB = 10.16 GB -> FITS
[run] 33 driving frames -> 1 segment(s), estimated 4-5 minutes of denoising

Stage plan:
  preprocess: python -m wan_t4.stages.preprocess --video /content/assets/sample_video.mp4 --image /content/assets/sample_face.png --work-dir outputs/work --resolution-area 480 480 --frame-num 33 --refert-num 1 --mode animate --max-frames 33
  text: python -m wan_t4.stages.text_encode --work-dir outputs/work --prompt a person dancing --negative-prompt 色调艳丽,过曝,静态,细节模糊不清,字幕,风格,作品,画作,画面,静止,整体发灰,最差质量,低质量,JPEG压缩残留,丑陋的,残缺的,多余的手指,画得不好的手部,画得不好的脸部,畸形的,毁容的,形态畸形的肢体,手指融合,静止不动的画面,杂乱的背景,三条腿,背景人很多,倒着走
  clip: python -m wan_t4.stages.clip_encode --work-dir outputs/work
  denoise: python -m wan_t4.stages.denoise --work-dir outputs/work --gguf weights/dit/Wan2.

## 8. Generar el video

Cada etapa corre su propio proceso. La memoria se restablece en cada etapa.

Si alguna etapa falla, los resultados previos no se pierden. Retomar con `--from denoise` (or la etapa que corresponda) para retomar desde los outputs parciales guardados en disco.

In [ ]:
!python -m wan_t4.run \
    --video "{VIDEO}" --image "{IMAGE}" \
    --prompt "{PROMPT}" --preset {PRESET} \
    --max-frames {MAX_FRAMES} \
    --out outputs/result.mp4

[run] preset 't4-draft' -> 480x480, 33f/segment, 4 steps, q3_k_m
[run] driving video: 3840x2160, 142 frames @ 24.0 fps
[run] 480x480 x 33f -> latents [10, 60, 60] = 9,000 tokens
[run] weights 8.04 GB + activations 0.58 GB + attention 0.34 GB = 10.16 GB -> FITS
[run] 33 driving frames -> 1 segment(s), estimated 4-5 minutes of denoising

=== stage: preprocess ===
$ /usr/bin/python3 -m wan_t4.stages.preprocess --video /content/assets/sample_video.mp4 --image /content/assets/sample_face.png --work-dir outputs/work --resolution-area 480 480 --frame-num 33 --refert-num 1 --mode animate --max-frames 33
[preprocess] reference 437x437 -> 480x480
[preprocess] driving 3840x2160 @ 24.00 fps -> 33 frames @ 30.00 fps
[preprocess] DWPose 288x384 + YOLOX 640x640 on CPUExecutionProvider (det batch 1, pose batch 4)
[preprocess] 9/33, ETA 42s frames (0.6 fps)
[preprocess] 18/33, ETA 26s frames (0.6 fps)
[preprocess] 25/33, ETA 14s frames (0.6 fps)
[preprocess] 33 frames in 57.3s (0.6 fps)
Multiple -pix_f

### Por ejemplo, si muere en la parte de "decode":


In [ ]:
!python -m wan_t4.run \
    --video "{VIDEO}" --image "{IMAGE}" \
    --prompt "{PROMPT}" --preset {PRESET} \
    --max-frames {MAX_FRAMES} \
    --out outputs/result.mp4 --from decode

## 9. Visualizar el resultado

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import pathlib

out = pathlib.Path('outputs/result.mp4')
assert out.exists(), 'no output produced -- check the stage logs above'
data = b64encode(out.read_bytes()).decode()
HTML(f'<video width=640 controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## Manejo de errores

**CUDA out of memory during denoising.** Reducir la carga en este orden:
1. Reducir `--frame-num` &mdash; aproximadamente lineal en costo.
2. Cambiar a `--dit-quant q3_k_m` si estabas en `q4_k_m` (8.0 GB vs 10.7 GB).
3. Reducir resolución &mdash; efectivo pero sacrifica más calidad.

**OOM during VAE decode, después de que terminó el denoising.** Los latentes ya están en disco. Puedes retomar la etapa siguiente usando `--from decode` (celda de ejemplo arriba).

**Output extraño, o el personaje no sigue el movimiento.** Revisa `outputs/work/src_pose.mp4` primero. Si el esqueleto está incorrecto o inexistente, el problema es extracción de pose, no el modelo de difusión. Se recomeinda cambiar el video.

**La cara se distorsiona.** Q3_K_M cuantiza el adaptador de rostros y todo lo demás. `befox/Wan2.2-Animate-14B-GGUF` publica variantes que mantienen el face-adapter a mayor precisión; puedes apuntar `--dit-quant` a uno de ellos.